# Model training: next-session return (Colab)

Trains and compares **zero-return baseline → Ridge → Random Forest → XGBoost** to predict
`next_return = price[t+1] / price[t] - 1`, then exports artifacts for the local app.

**Rules this notebook follows**
- Features come from the *same files the app imports* (`feature_schema.py`), so training and inference cannot drift.
- Chronological splits and walk-forward validation only. Nothing is shuffled. Imputers/scalers live inside the pipeline and are fit on training folds only.
- Model **selection uses walk-forward results on train+validation dates only**. The final test period is scored once, after selection.
- **No fabricated sentiment history.** Social/news features are used only if you supply a genuine, timestamped `historical_sentiment.csv`. Otherwise the model is **market-only** and is labelled that way.
- Ten days of freshly scraped posts cannot support a sentiment→price model; that is why training uses long OHLCV history.

**Before running:** upload `feature_schema.py`, `evaluation.py`, `trading_calendar.py` from the repo's `app/ml/` (cell 2 prompts you in Colab).

In [ ]:
!pip -q install xgboost yfinance scikit-learn pandas joblib

In [ ]:
# 2. Shared code: the exact modules the application imports
import pathlib
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SHARED = pathlib.Path("shared_ml") if IN_COLAB else pathlib.Path("../app/ml")
if IN_COLAB and not (SHARED / "feature_schema.py").exists():
    from google.colab import files
    SHARED.mkdir(exist_ok=True)
    print("Upload feature_schema.py, evaluation.py, trading_calendar.py (from app/ml/)")
    for name, data in files.upload().items():
        (SHARED / name).write_bytes(data)
sys.path.insert(0, str(SHARED.resolve()))

import evaluation as ev
import feature_schema as fs
from trading_calendar import TradingCalendar

print("feature schema", fs.FEATURE_SCHEMA_VERSION, "| market features:", len(fs.MARKET_FEATURES))

In [ ]:
# 3. Configuration (edit freely; keep tickers consistent with config/exchanges.json)
from datetime import date, time

SYMBOLS = {
    "HKEX": ["0700.HK", "9988.HK", "0005.HK", "1299.HK", "3690.HK"],
    "KRX":  ["005930.KS", "000660.KS", "005380.KS", "373220.KS", "068270.KS"],
}
CALENDARS = {  # used only to bucket timestamped sentiment into sessions (same class as the app)
    "HKEX": TradingCalendar("HKEX", "Asia/Hong_Kong", time(16, 10)),
    "KRX":  TradingCalendar("KRX", "Asia/Seoul", time(15, 30)),
}
START, END = "2016-01-01", date.today().isoformat()
SENTIMENT_CSV = None   # e.g. "historical_sentiment.csv" - see cell 6 for the required columns
N_SPLITS, MIN_TRAIN_DATES, SEED = 5, 250, 42

## 4. Historical OHLCV and a coverage check (nothing is assumed to exist)

In [ ]:
import numpy as np
import pandas as pd


def load_bars(symbol: str) -> pd.DataFrame:
    import yfinance as yf
    df = yf.Ticker(symbol).history(start=START, end=END, interval="1d", auto_adjust=False)
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.rename(columns={"Open": "open", "High": "high", "Low": "low", "Close": "close",
                             "Adj Close": "adj_close", "Volume": "volume"})
    out["session_date"] = pd.to_datetime(out.index).tz_localize(None).normalize()
    return out[["session_date", "open", "high", "low", "close", "adj_close", "volume"]].reset_index(drop=True)

# If a symbol is missing or poor, upload your own CSV (date,open,high,low,close,volume[,adj_close]) and load it here instead.
raw, rows = {}, []
for exch, syms in SYMBOLS.items():
    for s in syms:
        try:
            b = load_bars(s)
        except Exception as e:
            b = pd.DataFrame(); print("download failed", s, type(e).__name__, e)
        raw[s] = (exch, b)
        rows.append({"symbol": s, "bars": len(b), "first": b.session_date.min() if len(b) else None,
                     "last": b.session_date.max() if len(b) else None,
                     "missing_ohlc": int(b[["open", "high", "low", "close"]].isna().any(axis=1).sum()) if len(b) else None,
                     "zero_volume_days": int((b.volume == 0).sum()) if len(b) else None})
coverage = pd.DataFrame(rows)
display(coverage) if "display" in globals() else print(coverage)
usable = [s for s, (_, b) in raw.items() if len(b) >= 400]
print("usable symbols:", usable, "| excluded:", sorted(set(raw) - set(usable)))

## 5. Features and target
Built with `build_feature_frame`, the same function the app calls at inference. Rows where indicators are still warming up are dropped.

In [ ]:
frames = []
for s in usable:
    exch, b = raw[s]
    f = fs.add_target(fs.build_feature_frame(b.dropna(subset=["open", "high", "low", "close"])))
    f["symbol"], f["exchange"] = s, exch
    frames.append(f.iloc[fs.MIN_HISTORY_SESSIONS:])
data = pd.concat(frames, ignore_index=True)
data = data.dropna(subset=[fs.TARGET_NAME] + fs.MARKET_FEATURES).sort_values(["session_date", "symbol"]).reset_index(drop=True)
print(data.shape, data.session_date.min().date(), "->", data.session_date.max().date())
data[fs.TARGET_NAME].describe()

## 6. Optional: genuine historical sentiment
Supply `historical_sentiment.csv` **only if** it holds real, timestamped records collected at the time.
Columns: `symbol, event_time (UTC ISO), source_type (social|news), sentiment_score [-1,1], author_id, weight, engagement, relevance`.
Without it the model is **market-only** and no sentiment claim is made.

In [ ]:
FEATURE_SET = "market_only"
sentiment_coverage = None
if SENTIMENT_CSV:
    sent = pd.read_csv(SENTIMENT_CSV, parse_dates=["event_time"])
    sent["event_time"] = pd.to_datetime(sent["event_time"], utc=True)
    parts = []
    for s in usable:
        exch, b = raw[s]
        cal = CALENDARS[exch]
        r = sent[sent.symbol == s].copy()
        r["bucket_date"] = [pd.Timestamp(cal.bucket_date(t.to_pydatetime())) for t in r.event_time]
        f = fs.add_target(fs.build_feature_frame(b.dropna(subset=["open", "high", "low", "close"]),
                          r[r.source_type == "social"], r[r.source_type == "news"]))
        f["symbol"] = s
        parts.append(f.iloc[fs.MIN_HISTORY_SESSIONS:])
    full = pd.concat(parts, ignore_index=True).dropna(subset=[fs.TARGET_NAME] + fs.MARKET_FEATURES)
    covered = full[(full.social_has_data == 1) | (full.news_has_data == 1)]
    sentiment_coverage = {"rows_total": len(full), "rows_with_sentiment": len(covered),
                          "first_covered": str(covered.session_date.min().date()) if len(covered) else None,
                          "last_covered": str(covered.session_date.max().date()) if len(covered) else None}
    print("sentiment coverage:", sentiment_coverage)
    if len(covered) >= 600:
        data, FEATURE_SET = covered.sort_values(["session_date", "symbol"]).reset_index(drop=True), "market_sentiment"
        print("Training the SENTIMENT-ENHANCED experiment on covered sessions only.")
    else:
        print("Too little genuine sentiment coverage (<600 rows) -> staying market-only.")
else:
    print("No historical sentiment supplied -> MARKET-ONLY model.")
FEATURES = fs.FEATURE_SETS[FEATURE_SET]
print("feature set:", FEATURE_SET, len(FEATURES), "features")

## 7. Chronological split (train 60% / validation 20% / untouched test 20%)

In [ ]:
split = ev.chronological_split(data.session_date, 0.6, 0.2)
for k, (a, b) in split.items():
    print(f"{k:11} {pd.Timestamp(a).date()} -> {pd.Timestamp(b).date()}")
dev = data[data.session_date <= split["validation"][1]]    # used for model selection
test = data[data.session_date >= split["test"][0]]         # touched exactly once, after selection
print("dev rows", len(dev), "| test rows", len(test))

## 8. Candidates and walk-forward selection
Selection looks only at expanding-window walk-forward results on train+validation dates. Imputers/scalers are inside each pipeline, so they are refit per fold on training dates only.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

CANDIDATES = {
    "ridge": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0)),
    "random_forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
        RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=50, n_jobs=-1, random_state=SEED)),
    "xgboost": lambda: XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=5, reg_lambda=5.0, objective="reg:squarederror",
        tree_method="hist", random_state=SEED, n_jobs=1),
}
splits = ev.walk_forward_date_splits(dev.session_date, n_splits=N_SPLITS, min_train_dates=MIN_TRAIN_DATES)
wf = {}
for name, factory in CANDIDATES.items():
    p = ev.walk_forward_predict(factory, dev, FEATURES, fs.TARGET_NAME, "session_date", splits)
    wf[name] = ev.regression_metrics(p[fs.TARGET_NAME], p["prediction"])
base_p = p.copy(); base_p["prediction"] = 0.0
wf["zero_return_baseline"] = ev.regression_metrics(base_p[fs.TARGET_NAME], base_p["prediction"])
walk_forward_table = pd.DataFrame(wf).T
walk_forward_table

In [ ]:
model_candidates = {k: v for k, v in wf.items() if k != "zero_return_baseline"}
selected = min(model_candidates, key=lambda k: model_candidates[k]["mae"])
beats_baseline = wf[selected]["mae"] < wf["zero_return_baseline"]["mae"]
print("selected by walk-forward MAE:", selected, "| beats zero-return baseline:", beats_baseline)
if not beats_baseline:
    print("WARNING: no candidate beats the naive baseline out-of-sample. Report this honestly; do not market the model as predictive.")

## 9. Final fit on train+validation, single evaluation on the untouched test period

In [ ]:
final = CANDIDATES[selected]()
final.fit(dev[FEATURES], dev[fs.TARGET_NAME])
test_pred = final.predict(test[FEATURES])
test_out = test[["session_date", "symbol", fs.TARGET_NAME]].assign(prediction=test_pred)
m = ev.regression_metrics(test_out[fs.TARGET_NAME], test_out.prediction)
b = ev.regression_metrics(test_out[fs.TARGET_NAME], np.zeros(len(test_out)))
overall = {"model": m, "baseline_zero_return": b, **ev.baseline_improvement(m, b)}
per_symbol = {}
for s, g in test_out.groupby("symbol"):
    pm, pb = ev.regression_metrics(g[fs.TARGET_NAME], g.prediction), ev.regression_metrics(g[fs.TARGET_NAME], np.zeros(len(g)))
    per_symbol[s] = {"model": pm, "baseline_zero_return": pb, **ev.baseline_improvement(pm, pb)}
print(pd.json_normalize(overall).T)
pd.DataFrame({s: {"mae": v["model"]["mae"], "baseline_mae": v["baseline_zero_return"]["mae"],
                  "dir_acc": v["model"]["directional_accuracy"], "n": v["model"]["n"]} for s, v in per_symbol.items()}).T

## 10. Export artifacts (`joblib` + JSON metadata)

In [ ]:
import hashlib
import json
import platform
from datetime import UTC, datetime
from pathlib import Path

import joblib
import sklearn
import xgboost

OUT = Path("model_artifacts"); OUT.mkdir(exist_ok=True)
fname = "xgboost_model.joblib" if selected == "xgboost" else f"{selected}_model.joblib"
joblib.dump(final, OUT / fname)
iso = lambda t: pd.Timestamp(t).date().isoformat()
metadata = {
    "model_name": selected, "model_version": datetime.now(UTC).strftime("%Y%m%d-%H%M"),
    "training_date": datetime.now(UTC).isoformat(), "target": fs.TARGET_NAME, "horizon": fs.HORIZON,
    "feature_set": FEATURE_SET, "feature_columns": FEATURES, "feature_schema_version": fs.FEATURE_SCHEMA_VERSION,
    "allows_missing_values": True,                      # every candidate pipeline tolerates NaN
    "date_ranges": {k: [iso(a), iso(b_)] for k, (a, b_) in split.items()},
    "data_coverage": {"symbols": usable, "period": [iso(data.session_date.min()), iso(data.session_date.max())],
                      "bars_per_symbol": coverage.set_index("symbol")["bars"].to_dict(),
                      "historical_sentiment": sentiment_coverage},
    "validation_strategy": f"expanding-window walk-forward ({N_SPLITS} folds, min {MIN_TRAIN_DATES} dates) on train+validation; "
                           "final single evaluation on untouched chronological test period",
    "selected_by": "walk-forward MAE", "beats_zero_return_baseline_walk_forward": bool(beats_baseline),
    "package_versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                         "xgboost": xgboost.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "artifact_file": fname, "artifact_sha256": hashlib.sha256((OUT / fname).read_bytes()).hexdigest(),
    "limitations": "Daily next-session return model. Out-of-sample metrics describe the test period only and do not guarantee future performance.",
}
(OUT / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
(OUT / "feature_columns.json").write_text(json.dumps(FEATURES, indent=2))
(OUT / "evaluation_metrics.json").write_text(json.dumps(
    {"walk_forward": wf, "test": overall, "test_per_symbol": per_symbol}, indent=2, default=float))
print("exported:", sorted(p.name for p in OUT.iterdir()))
if fname != "xgboost_model.joblib":
    print(f"NOTE: set MODEL_PATH=model_artifacts/{fname} in the app's .env")

In [ ]:
# Download the four files, then place them in the repo's model_artifacts/ (see README).
if IN_COLAB:
    import shutil
    shutil.make_archive("model_artifacts", "zip", "model_artifacts")
    from google.colab import files
    files.download("model_artifacts.zip")